In [ ]:
import joblib 
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# pick which model to explain — comment out the others
TAG = "augmented"
# TAG = "qwen"; # TAG = "llama"
# TAG = "qwen_rewrite"; # TAG = "llama_rewrite"

b = joblib.load(f"xgb_bundle_{TAG}.joblib")
clf, tfidf         = b["clf"], b["tfidf"]
feat_names, hand_names = b["feat_names"], b["hand_names"]
Xte, te_x, te_y    = b["Xte"], b["te_x"], np.array(b["te_y"])
hte, htr, tr_y     = b["hte"], b["htr"], np.array(b["tr_y"])
prob, pred         = np.array(b["prob"]), np.array(b["pred"])
print(f"Loaded {TAG}: {Xte.shape[0]} test notes, {len(feat_names)} features")

# Q3: Why was this individual note flagged?

In [ ]:
import shap
explainer = shap.TreeExplainer(clf)
shap_values = explainer.shap_values(Xte)
base = explainer.expected_value

# a confidently-caught synthetic note, and a confidently-correct real note
syn_candidates  = np.where((te_y == 1) & (pred == 1))[0]
real_candidates = np.where((te_y == 0) & (pred == 0))[0]
syn_i  = int(syn_candidates[np.argmax(prob[syn_candidates])])
real_i = int(real_candidates[np.argmin(prob[real_candidates])])

for label, i in [("synthetic", syn_i), ("real", real_i)]:
    print(f"\n{'='*60}\n{label.upper()} note  (P_synthetic={prob[i]:.3f}, true={te_y[i]})\n{'='*60}")
    print(str(te_x[i])[:500], "...\n")
    shap.plots._waterfall.waterfall_legacy(
        base, shap_values[i], feature_names=feat_names, max_display=15, show=False)
    plt.title(f"{TAG} — {label} note"); plt.tight_layout()
    plt.savefig(f"q3_waterfall_{TAG}_{label}.png", dpi=150, bbox_inches="tight"); plt.show()